# Circuit Analysis and Electrodynamics: Equations, Variables, and Derivation

This notebook follows a textbook flow:
**field laws (Maxwell) -> constitutive relations -> circuit reduction -> Lagrangian/Hamiltonian form -> data-facing surrogate**.

Renderer note: Mermaid blocks depend on notebook renderer support. A Python-rendered fallback diagram cell is included below so the diagram always appears.

## Model Diagram (Field to Circuit to Data)
Mermaid source is kept for portability, but many notebook renderers do not embed Mermaid directly.
Run **Cell 2** to view the guaranteed rendered theory diagram.

## Full Maxwell System Used for Circuit Interpretation
For electrodynamics in matter:
1. $\nabla\cdot\mathbf{D}=\rho_f$.
2. $\nabla\cdot\mathbf{B}=0$.
3. $\nabla\times\mathbf{E}=-\partial\mathbf{B}/\partial t$.
4. $\nabla\times\mathbf{H}=\mathbf{J}_f+\partial\mathbf{D}/\partial t$.
5. Constitutive closure: $\mathbf{D}=\epsilon\mathbf{E}$, $\mathbf{B}=\mu\mathbf{H}$, $\mathbf{J}_f=\sigma\mathbf{E}$.

## Independent and Dependent Variables (and Why)
- Independent variables:
  - $t$ (time): required for dynamics and transients.
  - $(x,y,z)$ (space): required because Maxwell equations are field PDEs.
- Dependent variables:
  - Field level: $\mathbf{E}(x,y,z,t),\mathbf{H}(x,y,z,t),\mathbf{D}(x,y,z,t),\mathbf{B}(x,y,z,t)$.
  - Reduced circuit level: charge $q(t)$, current $i(t)=\dot q(t)$, flux linkage $\phi(t)$.
- Why this split:
  - Fields describe where energy is stored and transported.
  - Lumped variables are the low-dimensional projection used for fitting and control.

## Trivial Case Before Full Problem
We solve the lossless single-mode LC limit first:
1. $L\ddot q + \frac{1}{C}q=0$.
2. $\omega_0=1/\sqrt{LC}$.
3. $q(t)=Q_0\cos(\omega_0 t)$, $i(t)=\dot q(t)=-Q_0\omega_0\sin(\omega_0 t)$.
4. Energy check:
   $$
   \mathcal{E}(t)=\frac{q(t)^2}{2C}+\frac{L i(t)^2}{2}=\frac{Q_0^2}{2C}=\text{constant}.
   $$

The next code cell renders this trivial case both mathematically (printed identities) and graphically (time-series + energy conservation plot).

In [ ]:
# Theory diagram renderer (always visible in notebooks, independent of Mermaid support).
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.axis('off')
blocks = [
    (0.04, 0.68, 'Physical Assumptions'),
    (0.04, 0.40, 'Lagrangian / Governing PDE'),
    (0.04, 0.12, 'Reduced Dynamic Model'),
    (0.56, 0.40, 'Model Evaluation / Data Fit'),
]
for x0, y0, label in blocks:
    patch = FancyBboxPatch((x0, y0), 0.36, 0.18, boxstyle='round,pad=0.02', linewidth=1.2, facecolor='#eef4ff')
    ax.add_patch(patch)
    ax.text(x0 + 0.18, y0 + 0.09, label, ha='center', va='center', fontsize=10)
ax.annotate('', xy=(0.22, 0.62), xytext=(0.22, 0.50), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.annotate('', xy=(0.22, 0.34), xytext=(0.22, 0.22), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.annotate('', xy=(0.56, 0.49), xytext=(0.40, 0.21), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.set_title('Theory Diagram (Rendered Fallback)')
plt.tight_layout()
plt.show()

## Mechanics and Electrodynamics Bridge

### From Maxwell PDEs to a Lumped ODE
1. Start from Faraday and Ampere-Maxwell in integral form over device geometry.
2. Use mode/geometry assumptions to define a dominant generalized coordinate $q(t)$.
3. Obtain effective parameters $L$ and $C$ from magnetic/electric field energy.
4. Reduced equation: $L\ddot q + R\dot q + \frac{1}{C}q = v_s(t)$.

### Variational Form (Lossless Core)
1. Choose generalized coordinate $q$.
2. $T_{\text{mag}}=\frac{1}{2}L\dot q^2$.
3. $V_{\text{elec}}=\frac{1}{2C}q^2$.
4. Lagrangian: $\mathcal{L}(q,\dot q)=T_{\text{mag}}-V_{\text{elec}}$.
5. Euler-Lagrange gives $L\ddot q + \frac{1}{C}q=0$.

### Hamiltonian Form
1. Canonical momentum: $p=\partial\mathcal{L}/\partial\dot q=L\dot q$.
2. Hamiltonian:
   $$
   \mathcal{H}(q,p)=\frac{p^2}{2L}+\frac{q^2}{2C}.
   $$
3. Canonical equations recover the same oscillator dynamics.

### Data-Facing Variable Roles
1. Independent variable in fit cell: normalized time-like index $x\in[0,1]$.
2. Dependent measured variable: scalar observable $y(x)$ (voltage/current/proxy column from dataset).
3. Fitted parameters $(A,\gamma,\omega,c_0)$ map to amplitude, damping, frequency, and offset.
4. This surrogate is an observable approximation to the mechanistic state model above.

## Public Data Sources and Peer-Reviewed References

1. **UCI Electrical Grid Stability**
   - URL: https://archive.ics.uci.edu/ml/machine-learning-databases/00471/Data_for_UCI_named.csv
   - Peer-reviewed context: Arzamasov et al., Int. J. Electr. Power Energy Syst. (2018)
2. **Stock Data (CSV fallback)**
   - URL: https://raw.githubusercontent.com/plotly/datasets/master/stockdata.csv
   - Peer-reviewed context: Public benchmark fallback

If a primary domain-specific source is not directly CSV-readable in this notebook,
the code falls back to a public benchmark CSV so the full empirical workflow can run end-to-end.


In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import requests
import ipywidgets as widgets
from ipywidgets import interact

plt.style.use('seaborn-v0_8-whitegrid')

DOMAIN_KEY = '06_circuit_analysis'
DOMAIN_TITLE = 'Circuit Analysis and EM Fields'
MODEL_FAMILY = 'damped'
DATA_SOURCES = [
  {
    "name": "UCI Electrical Grid Stability",
    "url": "https://archive.ics.uci.edu/ml/machine-learning-databases/00471/Data_for_UCI_named.csv",
    "reference": "Arzamasov et al., Int. J. Electr. Power Energy Syst. (2018)"
  },
  {
    "name": "Stock Data (CSV fallback)",
    "url": "https://raw.githubusercontent.com/plotly/datasets/master/stockdata.csv",
    "reference": "Public benchmark fallback"
  }
]


In [ ]:
# Fallback model diagram renderer (works even when Mermaid is not embedded by the notebook renderer).
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.axis('off')
nodes = [
    (0.04, 0.68, 'Physical Assumptions'),
    (0.04, 0.40, 'Lagrangian / Governing PDE'),
    (0.04, 0.12, 'Reduced Dynamic Model'),
    (0.56, 0.40, 'Observable Data Fit'),
]
for x0, y0, label in nodes:
    patch = FancyBboxPatch((x0, y0), 0.36, 0.18, boxstyle='round,pad=0.02', linewidth=1.2, facecolor='#edf3fb')
    ax.add_patch(patch)
    ax.text(x0 + 0.18, y0 + 0.09, label, ha='center', va='center', fontsize=10)
ax.annotate('', xy=(0.22, 0.62), xytext=(0.22, 0.50), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.annotate('', xy=(0.22, 0.34), xytext=(0.22, 0.22), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.annotate('', xy=(0.56, 0.49), xytext=(0.40, 0.21), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.set_title('Fallback Diagram (Python-rendered)')
plt.tight_layout()
plt.show()

In [ ]:
# Trivial-case diagnostics for a lossless LC oscillator and field-to-circuit map.
from matplotlib.patches import FancyBboxPatch

L0 = 1.2
C0 = 0.8
Q0 = 1.0
omega0 = 1.0 / np.sqrt(L0 * C0)

t_demo = np.linspace(0.0, 25.0, 900)
q_demo = Q0 * np.cos(omega0 * t_demo)
i_demo = -Q0 * omega0 * np.sin(omega0 * t_demo)
energy_e = q_demo**2 / (2.0 * C0)
energy_m = 0.5 * L0 * i_demo**2
energy_total = energy_e + energy_m

print("Trivial LC case:")
print(f"L = {L0:.3f}, C = {C0:.3f}, omega0 = 1/sqrt(LC) = {omega0:.5f}")
print("q(t) = Q0 cos(omega0 t), i(t) = -Q0 omega0 sin(omega0 t)")
print(f"Energy drift (max-min): {np.max(energy_total) - np.min(energy_total):.3e}")

fig, ax = plt.subplots(1, 3, figsize=(15, 4))

# Simple diagram that always renders even when Mermaid is unavailable.
ax0 = ax[0]
ax0.axis("off")
boxes = [
    (0.05, 0.72, "Maxwell PDEs"),
    (0.05, 0.46, "Constitutive Laws"),
    (0.05, 0.20, "Lumped State q,i"),
    (0.58, 0.46, "Data Observable y(t)"),
]
for x0, y0, txt in boxes:
    patch = FancyBboxPatch((x0, y0), 0.35, 0.18, boxstyle="round,pad=0.02", linewidth=1.2, facecolor="#eef3ff")
    ax0.add_patch(patch)
    ax0.text(x0 + 0.175, y0 + 0.09, txt, ha="center", va="center", fontsize=10)
ax0.annotate("", xy=(0.225, 0.64), xytext=(0.225, 0.54), arrowprops=dict(arrowstyle="->", lw=1.3))
ax0.annotate("", xy=(0.225, 0.38), xytext=(0.225, 0.28), arrowprops=dict(arrowstyle="->", lw=1.3))
ax0.annotate("", xy=(0.58, 0.55), xytext=(0.40, 0.29), arrowprops=dict(arrowstyle="->", lw=1.3))
ax0.set_title("Model Diagram")

ax[1].plot(t_demo, q_demo, label="q(t): charge")
ax[1].plot(t_demo, i_demo, label="i(t): current")
ax[1].set_title("Trivial LC State Trajectories")
ax[1].set_xlabel("time t")
ax[1].set_ylabel("state value")
ax[1].legend()

ax[2].plot(t_demo, energy_e, label="electric energy q^2/(2C)")
ax[2].plot(t_demo, energy_m, label="magnetic energy L i^2/2")
ax[2].plot(t_demo, energy_total, "k--", linewidth=2.0, label="total")
ax[2].set_title("Energy Conservation (Lossless)")
ax[2].set_xlabel("time t")
ax[2].set_ylabel("energy")
ax[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
def try_read_public_csv(url: str):
    try:
        r = requests.get(url, timeout=20)
        r.raise_for_status()
        if 'text/csv' in r.headers.get('Content-Type', '') or ',' in r.text[:2000]:
            return pd.read_csv(io.StringIO(r.text))
    except Exception:
        return None
    return None

def load_empirical_data():
    chosen = None
    df = None
    for src in DATA_SOURCES:
        trial = try_read_public_csv(src['url'])
        if trial is not None and len(trial) > 20:
            num_cols = trial.select_dtypes(include=[np.number]).columns.tolist()
            if len(num_cols) >= 1:
                df = trial.copy()
                chosen = src
                break

    if df is None:
        x = np.linspace(0.0, 1.0, 600)
        y = np.exp(-2.5 * x) * np.cos(10.0 * x) + 0.05 * np.sin(40.0 * x)
        df = pd.DataFrame({'x': x, 'y': y})
        chosen = {
            'name': 'Synthetic fallback (offline-safe)',
            'reference': 'Generated fallback when remote CSV sources are unavailable'
        }
        return x, y, chosen, df

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) == 1:
        x = np.arange(len(df), dtype=float)
        y = df[numeric_cols[0]].to_numpy(dtype=float)
    else:
        x = df[numeric_cols[0]].to_numpy(dtype=float)
        y = df[numeric_cols[1]].to_numpy(dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if x.size < 8 or y.size < 8:
        x = np.linspace(0.0, 1.0, 600)
        y = np.exp(-2.5 * x) * np.cos(10.0 * x) + 0.05 * np.sin(40.0 * x)
        df = pd.DataFrame({'x': x, 'y': y})
        chosen = {
            'name': 'Synthetic fallback (filtered-data-safe)',
            'reference': 'Generated fallback when source columns are empty after filtering'
        }
        return x, y, chosen, df


    if len(x) > 2000:
        step = max(1, len(x) // 2000)
        x = x[::step]
        y = y[::step]

    if np.ptp(x) == 0:
        x = np.arange(len(x), dtype=float)

    x = (x - np.min(x)) / max(np.ptp(x), 1e-12)
    return x, y, chosen, df

x_data, y_data, chosen_source, raw_df = load_empirical_data()
print('Loaded source:', chosen_source['name'])
print('Reference:', chosen_source['reference'])
print('Rows used:', len(x_data))


In [ ]:
def model_function(x, p1, p2, p3, p4):
    if MODEL_FAMILY == 'oscillator':
        A, gamma, omega, c = p1, p2, p3, p4
        return A * np.exp(-gamma * x) * np.cos(omega * x) + c
    if MODEL_FAMILY == 'damped':
        A, gamma, _, c = p1, p2, p3, p4
        return A * np.exp(-gamma * x) + c
    if MODEL_FAMILY == 'logistic':
        K, r, x0, c = p1, p2, p3, p4
        return K / (1.0 + np.exp(-r * (x - x0))) + c
    if MODEL_FAMILY == 'michaelis':
        vmax, km, _, c = p1, p2, p3, p4
        return vmax * x / (km + x + 1e-12) + c
    if MODEL_FAMILY == 'diffusion':
        A, D, _, c = p1, p2, p3, p4
        return A * np.exp(-D * x) + c
    return p1 * x + p2

p0 = [np.nanmax(y_data) - np.nanmin(y_data), 0.5, 6.0, np.nanmean(y_data)]
bounds = ([-1e6, 0.0, -1e3, -1e6], [1e6, 50.0, 1e3, 1e6])

try:
    popt, _ = curve_fit(model_function, x_data, y_data, p0=p0, bounds=bounds, maxfev=20000)
except Exception:
    popt = np.array(p0, dtype=float)

y_hat = model_function(x_data, *popt)
rmse = float(np.sqrt(np.mean((y_data - y_hat) ** 2)))
r2 = float(1.0 - np.sum((y_data - y_hat) ** 2) / max(np.sum((y_data - np.mean(y_data)) ** 2), 1e-12))

print('Fitted parameters:', popt)
print('RMSE:', rmse)
print('R^2:', r2)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(x_data, y_data, s=12, alpha=0.5, label='Empirical data')
ax.plot(x_data, y_hat, color='crimson', linewidth=2.0, label='Fitted surrogate model')
ax.set_title(f'{DOMAIN_TITLE}: empirical fit ({MODEL_FAMILY})')
ax.set_xlabel('Normalized independent variable')
ax.set_ylabel('Observed variable')
ax.legend()
plt.show()


## High-Dimensional Hamiltonian Extension (Auto 3D/4D)

This block attempts to use the highest dimensional structure available from each project dataset.

1. If at least 4 numeric channels exist with enough rows, it builds a **data-derived 4D tensor** $(t,x,y,z)$.
2. If only 3 or 2 channels exist, it builds a 3D tensor then lifts to 4D with a latent mode axis.
3. If only 1 channel exists, it uses delay embedding and spatial basis lifting to create a 4D latent field.

Then it computes:
- Hamiltonian-energy proxy maps,
- modal singular spectrum and eigenfunction-like spatial modes,
- frequency-domain heatmaps in $(k_x,\omega)$, $(k_y,\omega)$, and $(k_r,\omega)$,
- 3D peak scatter in $(k_x,k_y,\omega)$.

In [ ]:
# Data-driven high-dimensional Hamiltonian analysis (auto 3D/4D with lifting for low-dimensional datasets).
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

rng = np.random.default_rng(0)

def _standardize(v):
    v = np.asarray(v, dtype=float)
    v = np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)
    mu = np.mean(v)
    sd = np.std(v)
    if sd < 1e-12:
        return v - mu
    return (v - mu) / sd

def _build_tensor_from_data(raw_df, x_data, y_data):
    if hasattr(raw_df, 'select_dtypes'):
        mat = raw_df.select_dtypes(include=[np.number]).to_numpy(dtype=float)
    else:
        mat = np.empty((0, 0), dtype=float)

    if mat.size == 0:
        mat = np.column_stack([x_data, y_data])

    mat = np.nan_to_num(mat, nan=0.0, posinf=0.0, neginf=0.0)
    n_rows, n_cols = mat.shape

    # Prefer true 4D data construction when possible.
    if n_cols >= 4 and n_rows >= 4 * 8 * 8 * 4:
        nt = min(24, max(8, n_rows // 256))
        nx = 8
        ny = 8
        nz = min(8, max(4, n_rows // (nt * nx * ny)))
        n_use = nt * nx * ny * nz
        base = np.mean(mat[:n_use, :4], axis=1)
        field = base.reshape(nt, nx, ny, nz)
        mode = '4D-from-data'
        return _standardize(field), mode

    # Build 3D from data, then lift to 4D via latent axial modes.
    if n_cols >= 3 and n_rows >= 8 * 8 * 8:
        nt = min(32, max(10, n_rows // 64))
        nx = 8
        ny = min(12, max(8, n_rows // (nt * nx)))
        n_use = nt * nx * ny
        base = np.mean(mat[:n_use, :3], axis=1).reshape(nt, nx, ny)
    elif n_cols >= 2:
        sig1 = _standardize(mat[:, 0])
        sig2 = _standardize(mat[:, 1])
        nt = min(36, max(12, len(sig1) // 8))
        nx, ny = 10, 10
        tt = np.linspace(0.0, 2.0 * np.pi, nt, endpoint=False)
        xx = np.linspace(0.0, 2.0 * np.pi, nx, endpoint=False)
        yy = np.linspace(0.0, 2.0 * np.pi, ny, endpoint=False)
        s1 = np.interp(np.linspace(0, len(sig1) - 1, nt), np.arange(len(sig1)), sig1)
        s2 = np.interp(np.linspace(0, len(sig2) - 1, nt), np.arange(len(sig2)), sig2)
        base = np.zeros((nt, nx, ny))
        for i in range(nt):
            base[i] = s1[i] * np.cos(xx)[:, None] + s2[i] * np.sin(yy)[None, :]
    else:
        sig = _standardize(y_data)
        nt = min(36, max(12, len(sig) // 8))
        nx, ny = 10, 10
        tt = np.linspace(0.0, 2.0 * np.pi, nt, endpoint=False)
        xx = np.linspace(0.0, 2.0 * np.pi, nx, endpoint=False)
        yy = np.linspace(0.0, 2.0 * np.pi, ny, endpoint=False)
        s = np.interp(np.linspace(0, len(sig) - 1, nt), np.arange(len(sig)), sig)
        base = np.zeros((nt, nx, ny))
        for i in range(nt):
            base[i] = s[i] * (np.cos(xx)[:, None] * np.cos(yy)[None, :])

    # Lift 3D base -> 4D (latent z modes).
    nz = 6
    z = np.linspace(0.0, 2.0 * np.pi, nz, endpoint=False)
    field = np.zeros((base.shape[0], base.shape[1], base.shape[2], nz))
    for k in range(nz):
        field[:, :, :, k] = base * (0.7 + 0.3 * np.cos((k + 1) * z[k])) + 0.08 * np.sin((k + 1) * base)
    mode = 'lifted-to-4D'
    return _standardize(field), mode

field4d, source_mode = _build_tensor_from_data(raw_df, x_data, y_data)
nt, nx, ny, nz = field4d.shape

# Canonical pair q, p from temporal derivative proxy.
q = field4d
p = np.gradient(q, axis=0)

# Hamiltonian density proxy at t0.
dq_dx = np.gradient(q[0], axis=0)
dq_dy = np.gradient(q[0], axis=1)
dq_dz = np.gradient(q[0], axis=2)
h_density = 0.5 * p[0]**2 + 0.5 * (dq_dx**2 + dq_dy**2 + dq_dz**2) + 0.5 * q[0]**2

# Modal decomposition (SVD of time-space unfolding).
X = q.reshape(nt, -1)
X = X - np.mean(X, axis=0, keepdims=True)
U, S, Vt = np.linalg.svd(X, full_matrices=False)
mode_energy = (S**2) / np.sum(S**2)
n_modes = min(4, Vt.shape[0])
modes = [Vt[i].reshape(nx, ny, nz) for i in range(n_modes)]

# 4D FFT -> power spectrum in (omega, kx, ky, kz).
P = np.abs(np.fft.fftn(q, axes=(0, 1, 2, 3)))**2
P = np.fft.fftshift(P, axes=(0, 1, 2, 3))
omega = np.fft.fftshift(2.0 * np.pi * np.fft.fftfreq(nt, d=1.0))
kx = np.fft.fftshift(2.0 * np.pi * np.fft.fftfreq(nx, d=1.0))
ky = np.fft.fftshift(2.0 * np.pi * np.fft.fftfreq(ny, d=1.0))
kz = np.fft.fftshift(2.0 * np.pi * np.fft.fftfreq(nz, d=1.0))

ikz0 = int(np.argmin(np.abs(kz)))
iky0 = int(np.argmin(np.abs(ky)))
P_kx_w = np.log10(np.sum(P[:, :, iky0, :], axis=2).T + 1e-12)
P_ky_w = np.log10(np.sum(P[:, :, :, ikz0], axis=1).T + 1e-12)

# Polar-radial spectrum in (kr, omega) from kx-ky projection.
P_w_kx_ky = np.sum(P, axis=3)
kxg, kyg = np.meshgrid(kx, ky, indexing='ij')
kr = np.sqrt(kxg**2 + kyg**2)
kr_flat = kr.ravel()
kr_bins = np.linspace(0.0, np.max(kr_flat), 24)
kr_centers = 0.5 * (kr_bins[:-1] + kr_bins[1:])
P_kr_w = np.zeros((omega.size, kr_centers.size))
for iw in range(omega.size):
    spec = P_w_kx_ky[iw].ravel()
    hist, _ = np.histogram(kr_flat, bins=kr_bins, weights=spec)
    P_kr_w[iw] = np.log10(hist + 1e-12)

# 3D spectral peak scatter in (kx, ky, omega) after summing kz.
Pk = np.sum(P, axis=3)
flat_idx = np.argpartition(Pk.ravel(), -180)[-180:]
iw, ikx, iky = np.unravel_index(flat_idx, Pk.shape)
peak_vals = np.log10(Pk[iw, ikx, iky] + 1e-12)

print(f'High-dimensional source mode: {source_mode}; tensor shape = {field4d.shape}')
print('Top modal energy fractions:', np.round(mode_energy[:8], 4))
print('Hamiltonian density stats: min=', float(h_density.min()), 'max=', float(h_density.max()), 'mean=', float(h_density.mean()))

# Figure A: Hamiltonian map + modal spectrum + eigenfunction-like slices
fig = plt.figure(figsize=(16, 12))
ax1 = fig.add_subplot(3, 3, 1)
im1 = ax1.imshow(q[0, :, :, nz // 2], origin='lower', cmap='coolwarm', aspect='auto')
ax1.set_title('q(t0, x, y, zmid)')
ax1.set_xlabel('y-index')
ax1.set_ylabel('x-index')
fig.colorbar(im1, ax=ax1, fraction=0.045)

ax2 = fig.add_subplot(3, 3, 2)
im2 = ax2.imshow(h_density[:, :, nz // 2], origin='lower', cmap='magma', aspect='auto')
ax2.set_title('Hamiltonian density slice at zmid')
ax2.set_xlabel('y-index')
ax2.set_ylabel('x-index')
fig.colorbar(im2, ax=ax2, fraction=0.045)

ax3 = fig.add_subplot(3, 3, 3)
ax3.plot(np.arange(1, min(21, len(mode_energy) + 1)), mode_energy[:20], marker='o', ms=3)
ax3.set_title('Modal energy spectrum (SVD)')
ax3.set_xlabel('mode index')
ax3.set_ylabel('energy fraction')

for i in range(min(4, n_modes)):
    ax = fig.add_subplot(3, 3, 4 + i)
    im = ax.imshow(modes[i][:, :, nz // 2], origin='lower', cmap='RdBu_r', aspect='auto')
    ax.set_title(f'Mode {i+1} slice at zmid')
    ax.set_xlabel('y-index')
    ax.set_ylabel('x-index')
    fig.colorbar(im, ax=ax, fraction=0.045)

plt.tight_layout()
plt.show()

# Figure B: Frequency-domain heatmaps
fig2, axes = plt.subplots(1, 3, figsize=(17, 4.5))
im_a = axes[0].imshow(P_kx_w, origin='lower', aspect='auto', extent=[kx.min(), kx.max(), omega.min(), omega.max()], cmap='viridis')
axes[0].set_title('log10 Power(kx, omega)')
axes[0].set_xlabel('kx')
axes[0].set_ylabel('omega')
fig2.colorbar(im_a, ax=axes[0], fraction=0.046)

im_b = axes[1].imshow(P_ky_w, origin='lower', aspect='auto', extent=[ky.min(), ky.max(), omega.min(), omega.max()], cmap='viridis')
axes[1].set_title('log10 Power(ky, omega)')
axes[1].set_xlabel('ky')
axes[1].set_ylabel('omega')
fig2.colorbar(im_b, ax=axes[1], fraction=0.046)

im_c = axes[2].imshow(P_kr_w.T, origin='lower', aspect='auto', extent=[omega.min(), omega.max(), kr_centers.min(), kr_centers.max()], cmap='plasma')
axes[2].set_title('log10 Power(kr, omega)')
axes[2].set_xlabel('omega')
axes[2].set_ylabel('kr')
fig2.colorbar(im_c, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.show()

# Figure C: 3D peak cloud
fig3 = plt.figure(figsize=(8, 6))
ax3d = fig3.add_subplot(111, projection='3d')
sc = ax3d.scatter(kx[ikx], ky[iky], omega[iw], c=peak_vals, s=10, cmap='inferno', alpha=0.8)
ax3d.set_title('Top spectral peaks in (kx, ky, omega)')
ax3d.set_xlabel('kx')
ax3d.set_ylabel('ky')
ax3d.set_zlabel('omega')
fig3.colorbar(sc, ax=ax3d, shrink=0.7, label='log10 power')
plt.tight_layout()
plt.show()

## Interpreting Model Evaluation Trends and Values

Use this guide when reading fitted parameters, RMSE, and $R^2$ outputs:

1. **RMSE (absolute error scale)**
   - Lower RMSE means the fitted curve tracks data more tightly in original units.
   - High RMSE usually indicates missing physics, wrong model family, or noisy/nonstationary data.

2. **$R^2$ (explained variance)**
   - Near 1.0: surrogate captures dominant trend/oscillation structure.
   - Near 0.0: surrogate explains little beyond the mean level.
   - Negative: model is worse than using a constant mean prediction.

3. **Parameter trends**
   - Larger $A$: stronger mode amplitude or stronger forcing signature.
   - Larger $\gamma$: faster decay / stronger damping / shorter memory.
   - Larger $\omega$: faster oscillation or higher characteristic frequency.
   - Offset $c_0$: baseline bias or slowly varying background level.

4. **Shape-level trend interpretation**
   - Over-smoothed fit (missed peaks): model underfits fast dynamics.
   - Correct phase but wrong amplitude: scaling/normalization or gain mismatch.
   - Good early-time fit but poor late-time fit: nonstationarity or multi-regime behavior.

5. **Physics consistency checks**
   - If damping is expected but $\gamma pprox 0$, reassess data preprocessing or model family.
   - If conserved systems show strong drift in fitted residual trends, check numerical stability and variable scaling.
   - Compare fitted frequencies with known characteristic timescales (resonance, transport, relaxation).

8. **High-dimensional modal/spectral interpretation**
   - Concentrated low-mode energy means coherent large-scale regimes dominate.
   - Fast modal-energy decay implies low-order model reduction is reliable.
   - Flat modal spectrum implies multi-scale coupling and weaker compressibility of dynamics.
   - Tight spectral peaks in $(k_x,k_y,\omega)$ imply persistent waves; broad clouds imply intermittency or transitions.
   - Drift of energy toward high $k_r$ indicates increasing fine-scale structure / cascade-like transfer.

## Physical Action and Measurement Simulation

Use the interactive controls below to manipulate physical parameters directly and observe how measured signals and energy maps change.

The controls are intentionally named after physical quantities (wave speed, damping, stiffness, forcing, diffusion, advection, measurement noise).

In [ ]:
# Interactive physical-action simulation with measurement outputs and physically named parameters.
from ipywidgets import interact, FloatSlider, IntSlider

def _prepare_base_field_for_simulation():
    if 'field4d' in globals() and isinstance(field4d, np.ndarray) and field4d.ndim == 4:
        base = np.array(field4d, dtype=float)
        return np.nan_to_num(base)
    if 'q' in globals() and isinstance(q, np.ndarray) and q.ndim == 4:
        base = np.array(q, dtype=float)
        return np.nan_to_num(base)

    # Fallback: lift available 1D measurement into a compact latent 4D field.
    sig = np.asarray(y_data if 'y_data' in globals() else np.sin(np.linspace(0.0, 8.0*np.pi, 480)), dtype=float)
    sig = np.nan_to_num(sig)
    if np.std(sig) > 1e-12:
        sig = (sig - np.mean(sig)) / np.std(sig)
    nt, nx, ny, nz = 36, 10, 10, 6
    tt = np.linspace(0.0, 2.0*np.pi, nt, endpoint=False)
    xx = np.linspace(0.0, 2.0*np.pi, nx, endpoint=False)
    yy = np.linspace(0.0, 2.0*np.pi, ny, endpoint=False)
    zz = np.linspace(0.0, 2.0*np.pi, nz, endpoint=False)
    s = np.interp(np.linspace(0, len(sig) - 1, nt), np.arange(len(sig)), sig)
    base = np.zeros((nt, nx, ny, nz), dtype=float)
    for i in range(nt):
        spatial = np.cos(xx)[:, None, None] * np.cos(yy)[None, :, None] * np.cos(zz)[None, None, :]
        base[i] = s[i] * spatial
    return base

BASE_FIELD = _prepare_base_field_for_simulation()
NT_SIM, NX_SIM, NY_SIM, NZ_SIM = BASE_FIELD.shape

def _laplacian_3d(arr3):
    return (
        np.roll(arr3, -1, axis=0) + np.roll(arr3, 1, axis=0)
        + np.roll(arr3, -1, axis=1) + np.roll(arr3, 1, axis=1)
        + np.roll(arr3, -1, axis=2) + np.roll(arr3, 1, axis=2)
        - 6.0 * arr3
    )

def physical_action_simulation(
    wave_speed,
    damping_coefficient,
    restoring_stiffness,
    forcing_amplitude,
    forcing_frequency,
    forcing_phase,
    nonlinear_coupling,
    diffusion_coefficient,
    advection_speed,
    measurement_noise_std,
    snapshot_time_index,
):
    base = BASE_FIELD
    nt, nx, ny, nz = base.shape
    t = np.linspace(0.0, 2.0*np.pi, nt, endpoint=False)
    tt = t[:, None, None, None]

    xx = np.linspace(0.0, 2.0*np.pi, nx, endpoint=False)
    yy = np.linspace(0.0, 2.0*np.pi, ny, endpoint=False)
    zz = np.linspace(0.0, 2.0*np.pi, nz, endpoint=False)
    forcing_pattern = np.sin(xx)[:, None, None] * np.cos(yy)[None, :, None] * np.sin(zz)[None, None, :]

    decay = np.exp(-damping_coefficient * tt)
    oscillatory = np.cos(wave_speed * tt + forcing_phase)
    q_linear = decay * (base * oscillatory)
    q_forcing = forcing_amplitude * np.sin(forcing_frequency * tt + forcing_phase) * forcing_pattern[None, :, :, :]

    q_nl = nonlinear_coupling * (base**3) / (1.0 + np.abs(base)**2)
    grad_x = np.gradient(base, axis=1)
    q_adv = -advection_speed * grad_x

    q_sim = q_linear + q_forcing + q_nl + q_adv

    # Apply diffusion and restoring effects per time slice.
    for i in range(nt):
        q_sim[i] = q_sim[i] + diffusion_coefficient * _laplacian_3d(q_sim[i]) - restoring_stiffness * q_sim[i]

    p_sim = np.gradient(q_sim, axis=0)

    # Measurement at a representative probe location.
    ix, iy, iz = nx // 2, ny // 2, nz // 2
    rng = np.random.default_rng(1234)
    measurement = q_sim[:, ix, iy, iz] + measurement_noise_std * rng.standard_normal(nt)

    # Energy density proxy at selected time.
    k = int(np.clip(snapshot_time_index, 0, nt - 1))
    dq_dx = np.gradient(q_sim[k], axis=0)
    dq_dy = np.gradient(q_sim[k], axis=1)
    dq_dz = np.gradient(q_sim[k], axis=2)
    energy_density = 0.5 * p_sim[k]**2 + 0.5 * (dq_dx**2 + dq_dy**2 + dq_dz**2) + 0.5 * restoring_stiffness * q_sim[k]**2

    # Frequency-domain measurement power.
    spec = np.fft.rfft(measurement - np.mean(measurement))
    freq = np.fft.rfftfreq(nt, d=1.0)
    power = np.abs(spec)**2

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    axes[0].plot(t, measurement, color='navy', lw=2)
    axes[0].set_title('Measured Signal at Probe')
    axes[0].set_xlabel('time')
    axes[0].set_ylabel('measurement')

    im = axes[1].imshow(energy_density[:, :, nz // 2], origin='lower', cmap='magma', aspect='auto')
    axes[1].set_title(f'Energy Density Slice (t index={k})')
    axes[1].set_xlabel('y-index')
    axes[1].set_ylabel('x-index')
    fig.colorbar(im, ax=axes[1], fraction=0.046)

    axes[2].plot(freq, power, color='crimson', lw=2)
    axes[2].set_title('Measurement Power Spectrum')
    axes[2].set_xlabel('frequency')
    axes[2].set_ylabel('power')

    plt.tight_layout()
    plt.show()

    print('Energy mean:', float(np.mean(energy_density)), '| Energy max:', float(np.max(energy_density)))
    print('Measurement std:', float(np.std(measurement)), '| Dominant frequency bin:', float(freq[np.argmax(power)]))

interact(
    physical_action_simulation,
    wave_speed=FloatSlider(value=1.3, min=0.2, max=4.0, step=0.05),
    damping_coefficient=FloatSlider(value=0.08, min=0.0, max=1.2, step=0.01),
    restoring_stiffness=FloatSlider(value=0.6, min=0.05, max=3.0, step=0.05),
    forcing_amplitude=FloatSlider(value=0.7, min=0.0, max=3.0, step=0.05),
    forcing_frequency=FloatSlider(value=1.8, min=0.1, max=8.0, step=0.05),
    forcing_phase=FloatSlider(value=0.0, min=-np.pi, max=np.pi, step=0.05),
    nonlinear_coupling=FloatSlider(value=0.15, min=-1.0, max=1.0, step=0.02),
    diffusion_coefficient=FloatSlider(value=0.06, min=0.0, max=0.5, step=0.01),
    advection_speed=FloatSlider(value=0.12, min=-1.0, max=1.0, step=0.02),
    measurement_noise_std=FloatSlider(value=0.02, min=0.0, max=0.5, step=0.01),
    snapshot_time_index=IntSlider(value=8, min=0, max=max(1, NT_SIM - 1), step=1),
)

## Interactive Simulation Controls

Use the sliders below to perturb model parameters and observe how well the
simulation matches the empirical measurements.


In [ ]:
def interactive_plot(p1, p2, p3, p4):
    y_sim = model_function(x_data, p1, p2, p3, p4)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.scatter(x_data, y_data, s=10, alpha=0.45, label='Empirical data')
    ax.plot(x_data, y_sim, color='navy', linewidth=2.0, label='Simulation')
    ax.set_title(f'{DOMAIN_TITLE}: interactive parameter sweep')
    ax.set_xlabel('Normalized independent variable')
    ax.set_ylabel('Observed variable')
    ax.legend()
    plt.show()

interact(
    interactive_plot,
    p1=widgets.FloatSlider(value=float(popt[0]), min=-2*abs(float(popt[0]))-1, max=2*abs(float(popt[0]))+1, step=0.01),
    p2=widgets.FloatSlider(value=float(popt[1]), min=0.0, max=max(1.0, 3*abs(float(popt[1]))+1), step=0.01),
    p3=widgets.FloatSlider(value=float(popt[2]), min=-20.0, max=20.0, step=0.05),
    p4=widgets.FloatSlider(value=float(popt[3]), min=float(np.min(y_data))-2, max=float(np.max(y_data))+2, step=0.01),
);
